# Navigable Small-World Graphs and the Mathematics of Greedy Routing — companion notebook

The **narrative** half of the Python pillar. The tested implementation lives next door in
[`navigable_small_world_graphs.py`](navigable_small_world_graphs.py); here we import it and walk the
topic section by section, so every claim renders as executed output.

The inverted file partitioned the space into cells; this topic replaces the flat partition with a
**graph**, and search becomes a walk. Two movements: first the *mathematics of greedy routing* —
Kleinberg's theorem that navigability requires a long-range link distribution matched to the
dimension — then the practical *navigable small-world graph* for approximate nearest-neighbor search.
The synthetic cloud is imported from the prerequisite high-dimensional-geometry topic.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "navigable-small-world-graphs",
              pathlib.Path("notebooks/navigable-small-world-graphs")):
    if (_cand / "navigable_small_world_graphs.py").exists():
        sys.path.insert(0, str(_cand)); break
import numpy as np
import navigable_small_world_graphs as nsw
X, queries = nsw.nsw_dataset()
adj, entry = nsw.build_nsw(X, nsw.NSW_M, nsw.NSW_EFC, seed=0)

## Movement 1 — the mathematics of greedy routing

### Kleinberg's navigability theorem

On a ring where each node has its two neighbors plus one long-range link drawn with probability
$\propto r^{-\alpha}$ (with $r$ the ring distance), decentralized greedy routing is fast **only** when
$\alpha$ equals the ring's dimension, $\alpha = 1$. Sweeping $\alpha$ traces a U-curve of mean
delivery time whose trough sits at the dimension; too uniform ($\alpha = 0$) or too local
($\alpha = 2$) and routing is far slower.

In [ ]:
for r in nsw.kleinberg_curve(nsw.RING_N, nsw.ALPHA_GRID):
    print(f"  alpha={r['alpha']:>4}: mean greedy hops = {r['hops']:.1f}")
nsw.test_kleinberg_navigability()

## Movement 2 — the navigable small-world graph

### Construction by incremental insertion, and the local-minimum catch

The graph is built by inserting points one at a time; each new point links to its $M$ approximate
nearest neighbors found by greedy search in the graph so far, and early insertions become long-range
hubs. At query time, **pure** greedy hill-climbing (beam width $\text{ef} = 1$) stops at a local
minimum, so its recall is below $1$ — the honest catch of graph search.

In [ ]:
nsw.test_greedy_local_minimum()

### The recall / work frontier

Widening the beam $\text{ef}$ recovers the missed neighbors monotonically, at the cost of touching
more vectors. Recall climbs to $1$ while the work stays a small fraction of $n$.

In [ ]:
for r in nsw.recall_vs_ef(X, queries, adj, entry, nsw.EF_GRID):
    print(f"  ef={r['ef']:>3}: recall={r['recall']:.3f}  distance computations={r['work']:.1f}")
nsw.test_recall_monotone_in_ef()

### Sublinear work and the small-world property

A moderate beam touches far fewer than $n$ vectors, and the constructed graph has a short average
path length — the long-range hubs make the diameter grow like $\log n$, not linearly.

In [ ]:
nsw.test_nsw_sublinear_work()
nsw.test_small_world_paths()

## Cross-check and viz constants

At a large beam width greedy search recovers nearly all true neighbors, and `viz_constants()` prints
the Kleinberg U-curve, the toy NSW graph and greedy walk, and the recall/work frontier — the exact
numbers the laboratory mirrors to the decimal.

In [ ]:
nsw.validate_nsw_high_ef()
nsw.viz_constants()

## All claims verified

Every theorem and the honest caveat are executed assertions above. The navigability U-curve, the
local-minimum recall, the recall/work frontier, and the small-world path length are the exact numbers
the laboratory mirrors. Re-run top to bottom to reproduce the topic.